/content/fg_interactive_budget


In [1]:
import pandas as pd
import os
from dotenv import load_dotenv



In [2]:
!git clone https://github.com/abelunbound/fg_interactive_budget.git

Cloning into 'fg_interactive_budget'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 258 (delta 146), reused 206 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (258/258), 12.78 MiB | 13.27 MiB/s, done.
Resolving deltas: 100% (146/146), done.


In [5]:
%cd /content/fg_interactive_budget

/content/fg_interactive_budget


In [ ]:
!pwd

In [11]:
# # Mandate data ingestion
input_path_csv = "/content/all_876_agencies_mandates.csv"

all_mandates = pd.read_csv(input_path_csv)


In [12]:
print(len(all_mandates))

876


In [13]:
all_mandates.head()

,mda_code,mda_name,mandate_description
0,521025001,"41. COMMUNITY HEALTH TUTOR PROGRAMME, UCH",The Community Health Tutor Programme at the Un...
1,517015001,42. COMPUTER PROFESSIONALS (REGISTRATION COUNC...,The Computer Professionals (Registration Counc...
2,119009118,"43. CONSULATE GENERAL OF NIGERIA, FRANKFURT, G...",The Consulate General of the Federal Republic ...
3,234005001,44. COUNCIL FOR THE REGULATION OF ENGINEERING ...,The Council for the Regulation of Engineering ...
4,229006001,45. COUNCIL FOR THE REGULATION OF FREIGHT FORW...,The Council for the Regulation of Freight Forw...


In [14]:
all_mandates.dtypes

,0
mda_code,int64
mda_name,object
mandate_description,object


In [24]:
ergp_budget_with_mda_path = 'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/approved_budget_2026.csv'

budget_df = pd.read_csv(ergp_budget_with_mda_path, nrows=500,
)





In [26]:
budget_df = budget_df.rename(
    columns={
        'mda_name_pdf': 'agency',
        'project': 'ergp_line_item',
        'type': 'status',
    }
)

In [22]:
print(len(budget_df))

500


In [27]:
budget_df.head()

,mda_code,agency,code,ergp_line_item,status,amount
0,111001001,STATE HOUSE - HQTRS,ERGP26102464,PURCHASE OF SPORTING EQUIPMENT FOR STATE HOUSE...,ONGOING,8.970837e+06
1,111001001,STATE HOUSE - HQTRS,ERGP26223957,PROCUREMENT /MAINTENANCE OF EQUIPMENT FOR THE\...,ONGOING,1.171335e+07
2,111001001,STATE HOUSE - HQTRS,ERGP27101865,RENOVATION WORK ON 8 NO. BLOCKS OF 16 NO. 2 B/...,ONGOING,8.419864e+07
3,111001001,STATE HOUSE - HQTRS,ERGP27201255,CONSTRUCTION OF OFFICE COMPLEX FOR Sas and SSAs,ONGOING,1.281548e+09
4,111001001,STATE HOUSE - HQTRS,ERGP27223750,ANNUAL ROUTINE MAINTENANCE OF MECHANICAL/ELECT...,ONGOING,4.229109e+09


In [28]:
budget_df.dtypes

,0
mda_code,int64
agency,object
code,object
ergp_line_item,object
status,object
amount,float64


In [30]:
# Remove commas and convert to float
budget_df['amount'] = budget_df['amount'].str.replace(',', '').astype('float64')

In [ ]:
budget_df.dtypes

code                object
ergp_line_item      object
status              object
amount             float64
mda_code             int64
agency              object
mother_ministry    float64
dtype: object

In [43]:
focus_mdas = budget_df['mda_code'].unique().tolist()


In [37]:
print(len(focus_mdas))

20


In [44]:
focus_mdas

[111001001,
 111001002,
 111001003,
 111001004,
 111001005,
 111001006,
 111001007,
 111006001,
 111007001,
 111009002,
 111010001,
 111011001,
 111048001,
 111056001,
 111058001,
 111062001,
 116001001,
 116002001,
 116002002,
 116003001]

In [45]:
focus_budget= budget_df[budget_df['mda_code'].isin(focus_mdas)]

In [46]:
print(len(focus_budget))

500


In [55]:
focus_mdas_df = all_mandates[all_mandates['mda_code'].isin(focus_mdas)]

In [56]:
print(len(focus_mdas_df))

20


In [47]:
focus_budget.dtypes

,0
mda_code,int64
agency,object
code,object
ergp_line_item,object
status,object
amount,float64


In [ ]:
# code	ergp_line_item	mda	amount	similarity_score	mandate_deviation_score



In [48]:
# Install anthropic if needed
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 11.3 MB/s eta 0:00:00


### LLM-AS-JUDGE:

In [59]:
# =============================================================================
# LLM-AS-JUDGE: CLAUDE API IMPLEMENTATION
# =============================================================================

import anthropic
import pandas as pd
import time
from typing import Optional


# # =============================================================================
# # CONFIGURATION
# # =============================================================================
# ANTHROPIC_API_KEY = os.getenv("CLAUDE_API_KEY")
# MODEL_NAME = "claude-sonnet-4-20250514"  # Deprecated #Good balance of speed/cost/quality
# MAX_RETRIES = 3
# RETRY_DELAY = 2  # seconds


# Import userdata to access Colab secrets
from google.colab import userdata
import os
# =============================================================================
# CONFIGURATION - Google Colab
# =============================================================================
ANTHROPIC_API_KEY = userdata.get('CLAUDE_API_KEY')  # Get from Colab secrets
MODEL_NAME = "claude-sonnet-4-6"
MAX_RETRIES = 3
RETRY_DELAY = 2  # seconds

In [60]:
# =============================================================================
# CORE FUNCTIONS
# =============================================================================

def create_client(api_key: str = ANTHROPIC_API_KEY) -> anthropic.Anthropic:
    """Create Anthropic client."""
    return anthropic.Anthropic(api_key=api_key)


def judge_alignment(
    client: anthropic.Anthropic,
    line_item: str,
    agency: str,
    mandate: str
) -> dict:
    """
    Use Claude to judge if budget item aligns with agency mandate.

    Returns:
        dict with keys: alignment, confidence, reason
    """
    prompt = f"""You are an auditor checking if Nigerian government budget items align with agency mandates.

    AGENCY: {agency}

    AGENCY MANDATE: {mandate}

    BUDGET LINE ITEM: {line_item}

    Does this budget item fall within the agency's mandate?

    Respond in EXACTLY this format (no other text):
    ALIGNMENT: YES or PARTIAL or NO
    CONFIDENCE: HIGH or MEDIUM or LOW
    REASON: (one sentence explanation)"""

    for attempt in range(MAX_RETRIES):
        try:
            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=150,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )

            result_text = response.content[0].text
            return parse_llm_response(result_text)

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                print(f"   Retry {attempt + 1}/{MAX_RETRIES} after error: {e}")
                time.sleep(RETRY_DELAY)
            else:
                print(f"   Failed after {MAX_RETRIES} attempts: {e}")
                return {
                    "alignment": "ERROR",
                    "confidence": "UNKNOWN",
                    "reason": str(e)
                }


def parse_llm_response(response: str) -> dict:
    """Parse the structured LLM response."""
    result = {
        "alignment": "UNKNOWN",
        "confidence": "UNKNOWN",
        "reason": ""
    }

    lines = response.strip().split("\n")

    for line in lines:
        line = line.strip()
        if line.upper().startswith("ALIGNMENT:"):
            value = line.split(":", 1)[1].strip().upper()
            if value in ["YES", "PARTIAL", "NO"]:
                result["alignment"] = value
        elif line.upper().startswith("CONFIDENCE:"):
            value = line.split(":", 1)[1].strip().upper()
            if value in ["HIGH", "MEDIUM", "LOW"]:
                result["confidence"] = value
        elif line.upper().startswith("REASON:"):
            result["reason"] = line.split(":", 1)[1].strip()

    return result


def compute_deviation_score(alignment: str, confidence: str) -> float:
    """
    Convert alignment + confidence to a deviation score (0-100).

    Higher score = more likely outside mandate = higher audit priority.
    """
    alignment_scores = {
        "YES": 0,
        "PARTIAL": 50,
        "NO": 100,
        "UNKNOWN": 50,
        "ERROR": 50
    }

    confidence_adjustments = {
        "HIGH": 0,
        "MEDIUM": 5,
        "LOW": 10,
        "UNKNOWN": 10
    }

    base_score = alignment_scores.get(alignment, 50)
    adjustment = confidence_adjustments.get(confidence, 10)

    # For YES: add adjustment (less confident YES = slightly higher deviation)
    # For NO: subtract adjustment (less confident NO = slightly lower deviation)
    if alignment == "YES":
        return min(base_score + adjustment, 100)
    elif alignment == "NO":
        return max(base_score - adjustment, 0)
    else:
        return base_score


# =============================================================================
# MAIN ANALYSIS FUNCTION
# =============================================================================

def analyze_budget_mandate_alignment_llm(
    budget_df: pd.DataFrame,
    mandate_df: pd.DataFrame,
    api_key: str = ANTHROPIC_API_KEY,
    progress_interval: int = 10
) -> pd.DataFrame:
    """
    Analyze alignment between budget items and agency mandates using Claude.

    Args:
        budget_df: DataFrame with columns [code, ergp_line_item, mda_code, agency, mother_ministry, amount]
        mandate_df: DataFrame with columns [mda_code, mda_name, mandate_description]
        api_key: Anthropic API key
        progress_interval: Print progress every N items

    Returns:
        DataFrame with alignment results
    """
    # Create client
    client = create_client(api_key)

    # Create mandate lookup by mda_code
    mandate_lookup = dict(zip(mandate_df["mda_code"], mandate_df["mandate_description"]))

    results = []
    total_items = len(budget_df)

    print(f"Analyzing {total_items} budget items with Claude...")
    print(f"Estimated time: {total_items * 1.5 / 60:.1f} - {total_items * 2.5 / 60:.1f} minutes")
    print("=" * 60)

    start_time = time.time()

    for i, (_, row) in enumerate(budget_df.iterrows()):
        mda_code = row["mda_code"]
        line_item = row["ergp_line_item"]
        agency = row["agency"]
        mandate = mandate_lookup.get(mda_code, "")

        if mandate:
            judgement = judge_alignment(client, line_item, agency, mandate)
        else:
            judgement = {
                "alignment": "UNKNOWN",
                "confidence": "UNKNOWN",
                "reason": "No mandate found for this agency"
            }

        deviation = compute_deviation_score(
            judgement["alignment"],
            judgement["confidence"]
        )

        results.append({
            "code": row["code"],
            "ergp_line_item": row["ergp_line_item"],
            "mda_code": row["mda_code"],
            "agency": row["agency"],
            # "mother_ministry": row["mother_ministry"],
            "amount": row["amount"],
            "alignment": judgement["alignment"],
            "confidence": judgement["confidence"],
            "reason": judgement["reason"],
            "mandate_deviation_score": deviation
        })

        # Progress update
        if (i + 1) % progress_interval == 0 or (i + 1) == total_items:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            remaining = (total_items - i - 1) / rate if rate > 0 else 0
            print(f"   Processed {i + 1}/{total_items} ({(i+1)/total_items*100:.1f}%) | "
                  f"Elapsed: {elapsed/60:.1f}min | Remaining: {remaining/60:.1f}min")

    print("=" * 60)
    print("Analysis complete.")

    return pd.DataFrame(results)




In [61]:
# =============================================================================
# OUTPUT & REPORTING
# =============================================================================

def print_summary_llm(result_df: pd.DataFrame) -> None:
    """Print summary statistics for LLM analysis."""
    print("\n" + "=" * 80)
    print("SUMMARY: LLM-AS-JUDGE ANALYSIS")
    print("=" * 80)

    total_items = len(result_df)
    total_amount = result_df["amount"].sum()

    # Alignment distribution
    alignment_counts = result_df["alignment"].value_counts()

    print(f"\nTotal Budget Items: {total_items}")
    print(f"Total Budget Amount: ₦{total_amount:,.0f}")

    print(f"\nAlignment Distribution:")
    for alignment, count in alignment_counts.items():
        pct = count / total_items * 100
        amount = result_df[result_df["alignment"] == alignment]["amount"].sum()
        print(f"   {alignment}: {count} items ({pct:.1f}%) | ₦{amount:,.0f}")

    # Flagged items (NO alignment)
    flagged = result_df[result_df["alignment"] == "NO"]
    print(f"\n🚨 Items flagged as NOT aligned: {len(flagged)}")
    print(f"   Flagged amount: ₦{flagged['amount'].sum():,.0f}")


def print_flagged_items_llm(result_df: pd.DataFrame, top_n: int = 20) -> None:
    """Print items flagged as not aligned."""
    flagged = result_df[result_df["alignment"] == "NO"].sort_values(
        "amount", ascending=False
    ).head(top_n)

    if len(flagged) == 0:
        print("\n✅ No items flagged as misaligned!")
        return

    print("\n" + "=" * 100)
    print(f"TOP {top_n} FLAGGED ITEMS (NOT ALIGNED)")
    print("=" * 100)

    for _, row in flagged.iterrows():
        print(f"\n🚨 {row['code']}: {row['ergp_line_item'][:60]}...")
        print(f"   Agency: {row['agency']}")
        print(f"   Amount: ₦{row['amount']:,.0f}")
        print(f"   Confidence: {row['confidence']}")
        print(f"   Reason: {row['reason']}")



In [62]:
# =============================================================================
#  USAGE
# =============================================================================


# Example usage (replace with your actual data)

# 1. Set your API key
api_key = ANTHROPIC_API_KEY

# 2. Run analysis
result_df = analyze_budget_mandate_alignment_llm(
    budget_df=budget_df,
    mandate_df=focus_mdas_df,
    api_key=api_key,
    progress_interval=10
)



Analyzing 500 budget items with Claude...
Estimated time: 12.5 - 20.8 minutes
   Processed 10/500 (2.0%) | Elapsed: 0.4min | Remaining: 21.4min
   Processed 20/500 (4.0%) | Elapsed: 0.8min | Remaining: 20.2min
   Processed 30/500 (6.0%) | Elapsed: 1.2min | Remaining: 19.4min
   Processed 40/500 (8.0%) | Elapsed: 1.7min | Remaining: 19.3min
   Processed 50/500 (10.0%) | Elapsed: 2.0min | Remaining: 18.2min
   Processed 60/500 (12.0%) | Elapsed: 2.4min | Remaining: 17.6min
   Processed 70/500 (14.0%) | Elapsed: 2.9min | Remaining: 17.6min
   Processed 80/500 (16.0%) | Elapsed: 3.3min | Remaining: 17.1min
   Processed 90/500 (18.0%) | Elapsed: 3.7min | Remaining: 16.9min
   Processed 100/500 (20.0%) | Elapsed: 4.2min | Remaining: 16.6min
   Processed 110/500 (22.0%) | Elapsed: 4.8min | Remaining: 16.9min
   Processed 120/500 (24.0%) | Elapsed: 5.2min | Remaining: 16.5min
   Processed 130/500 (26.0%) | Elapsed: 5.7min | Remaining: 16.1min
   Processed 140/500 (28.0%) | Elapsed: 6.1min | Re

In [63]:
print(len(result_df))

500


In [64]:
result_df.columns.to_list()

['code',
 'ergp_line_item',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'confidence',
 'reason',
 'mandate_deviation_score']

In [65]:
result_df.dtypes

,0
code,object
ergp_line_item,object
mda_code,int64
agency,object
amount,float64
alignment,object
confidence,object
reason,object
mandate_deviation_score,int64


In [67]:
# 3. View results
print_summary_llm(result_df)
print_flagged_items_llm(result_df)




SUMMARY: LLM-AS-JUDGE ANALYSIS

Total Budget Items: 500
Total Budget Amount: ₦257,496,574,655

Alignment Distribution:
   NO: 224 items (44.8%) | ₦103,354,645,700
   YES: 168 items (33.6%) | ₦139,037,102,572
   PARTIAL: 108 items (21.6%) | ₦15,104,826,383

🚨 Items flagged as NOT aligned: 224
   Flagged amount: ₦103,354,645,700

TOP 20 FLAGGED ITEMS (NOT ALIGNED)

🚨 ERGP20269504: INFRASTRUCTURE DEVELOPMENT...
   Agency: BUREAU OF PUBLIC PROCUREMENT (BPP)
   Amount: ₦20,000,000,000
   Confidence: HIGH
   Reason: Infrastructure development is outside BPP's mandate, which is strictly focused on regulating, monitoring, and overseeing procurement processes; actual infrastructure development is the responsibility of line ministries and implementing agencies, not the procurement regulatory body.

🚨 ERGP16234836: "ACQUISITION OF BUREAU OF PUBLIC PROCUREMENT
HEADQUARTER OFF...
   Agency: BUREAU OF PUBLIC ENTERPRISES (BPE)
   Amount: ₦14,000,000,000
   Confidence: HIGH
   Reason: This budget ite

In [68]:
# 4. Export
result_df.to_csv("/content/mandate_alignment_500_test.csv", index=False)

pass

In [ ]:
result_df.columns.to_list()

['code',
 'ergp_line_item',
 'mda_code',
 'agency',
 'mother_ministry',
 'amount',
 'alignment',
 'confidence',
 'reason',
 'mandate_deviation_score']

In [ ]:
print(len(result_df))

5686


In [ ]:
result_df.dtypes

code                        object
ergp_line_item              object
mda_code                     int64
agency                      object
mother_ministry            float64
amount                     float64
alignment                   object
confidence                  object
reason                      object
mandate_deviation_score      int64
dtype: object